# MC++ ellipsoidal image pybind11 test

This notebook reproduces the C++ `ELL-Doxygen.cpp` test using the Python interface exposed by `ellimage.cpp`.

It checks that the lifted ellipsoidal image has dimension 8, that the lifted shape matrix is PSD, and that projecting the two nonlinear `EllVar` expressions gives a 2D PSD ellipsoid.

## Import the compiled pybind11 module

In [ ]:
import numpy as np
import pymc

## Helper functions

These mirror the small C++ `require(...)` and `psd(...)` helpers.

In [ ]:
def require(ok: bool, msg: str) -> None:
    """Python equivalent of the small C++ require(...) helper."""
    if not ok:
        raise AssertionError(msg)


def psd(Q, tol: float = 1e-8) -> bool:
    """Return True when a symmetric matrix is positive semidefinite to tolerance."""
    Q = np.asarray(Q, dtype=float)
    eig = np.linalg.eigvalsh(0.5 * (Q + Q.T))
    return bool(np.min(eig) >= -tol)


def show_matrix(name, M):
    print(f"{name}.shape = {np.asarray(M).shape}")
    print(np.asarray(M, dtype=float))

## Reproduce `ELL-Doxygen.cpp`

In [ ]:
# Reproduce the documented C++ example with the current default EllImg options.
pymc.EllImg.options = pymc.EllImg.Options()

cx = np.array([3.0, 4.0], dtype=float)
Qx = np.array(
    [
        [5.0, 4.0],
        [4.0, 5.0],
    ],
    dtype=float,
)

Ex = pymc.EllImg(Qx, cx)
X1 = pymc.EllVar(Ex, 0)
X2 = pymc.EllVar(Ex, 1)

F = [
    pymc.log(X1) + pymc.sqr(X2),
    pymc.sin(X1) - pymc.cos(X2),
]

qlift = np.asarray(Ex.c_lift, dtype=float)
Qlift = np.asarray(Ex.Q_lift, dtype=float)

print("Lifted image:")
show_matrix("Q_lift", Qlift)
print("c_lift =", qlift)

require(qlift.size == 8, "documented example lifted dimension")
require(Qlift.shape == (8, 8), "documented example lifted shape size")
require(psd(Qlift), "documented example lifted shape is PSD")

Ef = Ex.get(F)
print("\nProjected ellipsoid Ef:")
print(Ef)

require(Ef.n == 2, "documented example projection dimension")
require(psd(Ef.Q), "documented example projected shape is PSD")

print("\nellimage Doxygen example test passed.")